# Delegation check -- demonstrate the 4th threat axis's blind spot (CPO / eMSP / EV)

**This notebook has been trimmed.** Generating the delegation columns (writing `delegation.py`, wiring it into `multi_world_pipeline.py`, clearing stale checkpoints, and regenerating all 7 builds) now all happens in `02_generate_v2g_benchmark.ipynb` itself -- **run that notebook first.** This notebook only does the detection/check side: it loads the resulting `v2g_benchmark_multi_world.csv` from Drive and demonstrates the blind spot. It is otherwise self-contained (it writes its own copies of `naive_trust.py` and `delegation_baseline.py` below), so you do not need notebook 04 open in the same runtime.

Recap of the three delegation scenarios this checks (full design/rationale lives in `delegation.py`'s own docstring, written by notebook 02):

- **CPO delegation** -- a CPO (`CPO-A`, one of the honest ones) sub-contracts 15% of its stations to a delegate operator (`CPO-D-sub`). The delegate commits the same kind of billing fraud as the already-known malicious CPO (`CPO-B`), but it is keyed on a NEW `station_operator_id` column, not on `cpo_id`. `CPO-A`'s own `authority_malicious` ground truth stays 0 -- that is the blind spot: a system that only ever tracks `cpo_id` never sees this fraud at all.
- **eMSP delegation** -- on roaming sessions, credential issuance is sometimes delegated to a visited-network hub (`eMSP-D-roam`, a NEW `issuing_emsp_id` column) instead of the driver's home eMSP handling it itself. The delegate hub has its own elevated dispute rate, tracked separately from the home eMSP's own dispute history.
- **EV delegation** -- 10% of sessions are driven by someone other than the registered owner (`delegate_driver_id`, a NEW column) -- mostly one-off and benign (family, valet), except for a small pool of 5 recurring malicious delegate identities who reuse access across MANY different owner accounts. That reuse-across-accounts pattern, not raw session count, is the actual abuse signature -- distinct from the existing `ev_sybil` axis.

For each axis, this compares the EXISTING home-identity-only naive trust (`naive_cpo_trust` / `naive_emsp_trust` / `naive_ev_trust`, from `naive_trust.py` -- the same module already used in notebook 04) against a NEW delegation-aware view keyed on the actual acting delegate. If the home-identity view still rates delegate-run fraud as highly trustworthy while the delegation-aware view does not, that is the blind spot the paper can report -- the same shape of finding as the existing single-vs-multi-authority story in notebook 04, but for identity delegation instead of authority coverage. This runs on the FULL combined dataset (not a held-out test split), for simplicity -- it shows the shape of the blind spot per variant, not a rigorous train/test ML evaluation like notebook 05.


## Step 1: write the trust-baseline modules this check needs

`naive_trust.py` (unchanged from notebook 04) and `delegation_baseline.py` (new, the detection side of the delegation axis) are written fresh here so this notebook doesn't depend on notebook 04 having been opened in this same runtime.

In [ ]:
%%writefile naive_trust.py
"""
naive_trust.py
----------------
Naive (unfused) per-authority trust trajectories for the ABAC baseline --
deliberately NOT using TrustMesh's own Algorithm 1 (fuzzy engine,
fuzzy_trust.py) or Algorithm 2 (subjective-logic propagation,
subjective_logic.py), so the ABAC baseline stays independent of
TrustMesh rather than secretly running it underneath (TrustMesh feeding
its OWN trust into an ABAC policy layer is a separate, already-published
result -- see trustmesh_pkg/run_experiments.py's Experiment 6). Each
function here is a plain, CAUSAL (strictly-prior-session, no look-ahead)
running statistic -- nothing in this module looks at a row's own future.

GROUNDING LEDGER:
  - naive_cpo_trust: a LINEAR (not fuzzy-membership) rescaling of the
    causal running mean of billing_deviation for that CPO, against a
    SCALE calibrated from the honest population (same calibrate-from-
    honest-population discipline as fuzzy_trust.py, just linear instead
    of triangular-membership, and deliberately calibrated from the
    TRAIN split only -- see compute_cpo_trust_scale -- so the test split
    never influences the calibration constant).
  - naive_emsp_trust: 1 - previous_flag_rate. previous_flag_rate ALREADY
    EXISTS as a column (enriched_features.py) -- reused directly, not
    recomputed: it's already exactly this, a causal, prior-only running
    dispute rate per eMSP.
  - naive_ev_trust: NEW. A causal running "session-share" statistic per
    EV -- the fraction of all sessions seen so far that belong to this
    EV, compared against the uniform-usage expectation
    (1 / N_SYNTHETIC_EVS). A Sybil EV's defining signature (see
    enriched_features.inject_ev_sybil) is reusing the SAME identity far
    more than organic usage would -- this is a direct, simple read on
    exactly that, with no graph structure needed (the graph-structural
    baseline gets that machinery; this is deliberately the plainer,
    non-structural comparison point available to a policy engine that
    has no graph layer at all).

BUG LOG (found while wiring up the 7-build variant suite, confirmed
PRE-EXISTING -- present even for the original single-build, 2-world
"main" dataset, not something the variant work introduced): both
add_naive_cpo_trust and add_naive_ev_trust sort the ENTIRE passed-in
dataframe by connection_time and keep one running_sum/running_n dict
per cpo_id/ev_id, with no awareness of which world or variant a row
belongs to. World 0 and world 1 (and, now, all 7 variants) are built
from the SAME real ACN-Data connection_time values -- only the
synthetic identity/fraud/feature draws differ -- so two rows that are
genuinely independent synthetic universes can carry the IDENTICAL
timestamp. Sorting the union by connection_time interleaves them, and
a tied timestamp breaks ties by original row order (world 0 rows were
concatenated before world 1's in multi_world_pipeline.py), so world
1's running trust for "CPO-B" silently started from wherever world 0's
running sum happened to be instead of a clean cold start -- the same
leak, now across 14 (variant, world) groups instead of 2 plain worlds.
Fixed by grouping the causal computation by (variant, world_id) before
running each entity's running statistic, falling back to no grouping
when those columns aren't present (e.g. a single-world/single-variant
subset passed in directly) so existing single-build callers are
unaffected. add_naive_emsp_trust is NOT affected: it just reads
previous_flag_rate, which enriched_features.py already computes
per-world at generation time, before any combining.
"""

import numpy as np
import pandas as pd

N_SYNTHETIC_EVS = 1000  # mirrors identity_assignment.py
EV_SHARE_SUSPICION_MULTIPLE = 5.0  # documented assumption: an EV using more
                                     # than 5x its uniform-expected share of
                                     # sessions seen so far is treated as
                                     # fully untrustworthy (naive_ev_trust ->
                                     # 0); linear in between, 0 share -> 1.0

DEFAULT_GROUP_COLS = ("variant", "world_id")  # each (variant, world_id) pair is
# an independent synthetic universe sharing the same real connection_time
# range as every other one -- causal running statistics must never cross
# this boundary (see BUG LOG above).


def _present_group_cols(df: pd.DataFrame, group_cols) -> list:
    return [c for c in group_cols if c in df.columns]


def _iter_groups(df: pd.DataFrame, group_cols):
    """Yields (key, sub_df) pairs to compute a causal running statistic
    independently within. Falls back to the whole frame as one group when
    none of group_cols are present, so single-world/single-variant callers
    (or a caller that already sliced to one world) see no behavior change."""
    cols = _present_group_cols(df, group_cols)
    if not cols:
        yield None, df
        return
    for key, g in df.groupby(cols, sort=False):
        yield key, g


def compute_cpo_trust_scale(train_df: pd.DataFrame, malicious_cpo: str,
                               quantile: float = 0.999, per_variant: bool = True):
    """Calibration constant for add_naive_cpo_trust, derived ONLY from the
    TRAIN split's apparently-honest rows (cpo_id != malicious_cpo) -- never
    from test, so the one calibrated number in this module can't leak
    test-period information into the baseline.

    When train_df spans more than one variant and per_variant=True (the
    default), returns a DICT {variant_name: scale} instead of a single
    float: noisy_benign's honest population is deliberately noisier than
    main's, so calibrating every variant off one pooled quantile would
    judge noisy_benign's honest sessions against a scale that's too tight
    (leaked from the other, quieter variants) and make them look less
    trustworthy than main's honest sessions for no behavioral reason.
    Pass per_variant=False, or call on a single-variant frame, to get the
    original single-float behavior back."""
    if per_variant and "variant" in train_df.columns:
        out = {}
        for vname, g in train_df.groupby("variant", sort=False):
            honest = g.loc[g["cpo_id"] != malicious_cpo, "billing_deviation"]
            s = float(honest.quantile(quantile))
            out[vname] = s if s > 1e-9 else 1e-6
        return out
    honest = train_df.loc[train_df["cpo_id"] != malicious_cpo, "billing_deviation"]
    scale = float(honest.quantile(quantile))
    return scale if scale > 1e-9 else 1e-6


def add_naive_cpo_trust(df: pd.DataFrame, scale, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """df should be the FULL chronological sequence (all worlds/rows you want
    trajectories over) -- causal, so safe to run across train+test combined;
    only `scale` (from compute_cpo_trust_scale) must come from train alone.

    `scale` is either a single float (applied to every row) or a dict
    {variant_name: float} (see compute_cpo_trust_scale) -- resolved per
    group below. The running statistic itself is computed independently
    within each (variant, world_id) group (see module BUG LOG): grouping
    first, then sorting each group by connection_time, keeps every group's
    running sums starting from a clean cold start instead of leaking into
    each other through shared/tied real timestamps."""
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)
    for key, g in _iter_groups(df, group_cols):
        if isinstance(scale, dict):
            variant_cols = _present_group_cols(df, ("variant",))
            this_scale = scale[g["variant"].iloc[0]] if variant_cols else next(iter(scale.values()))
        else:
            this_scale = scale

        order = g.sort_values("connection_time").index
        bd = g.loc[order, "billing_deviation"].to_numpy()
        cpo = g.loc[order, "cpo_id"].to_numpy()

        running_sum: dict = {}
        running_n: dict = {}
        local = np.empty(len(g))
        for i, c in enumerate(cpo):
            n = running_n.get(c, 0)
            running_mean = (running_sum.get(c, 0.0) / n) if n > 0 else 0.0  # cold start: assume trustworthy
            local[i] = float(np.clip(1.0 - running_mean / this_scale, 0.0, 1.0))
            running_sum[c] = running_sum.get(c, 0.0) + bd[i]
            running_n[c] = n + 1
        trust.loc[order] = local

    df["naive_cpo_trust"] = trust
    return df


def add_naive_emsp_trust(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["naive_emsp_trust"] = 1.0 - df["previous_flag_rate"]
    return df


def add_naive_ev_trust(df: pd.DataFrame, n_evs: int = N_SYNTHETIC_EVS,
                          suspicion_multiple: float = EV_SHARE_SUSPICION_MULTIPLE,
                          group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    """See module BUG LOG: the running EV session-share statistic is
    computed independently within each (variant, world_id) group, not
    across the full passed-in frame, for the same reason as
    add_naive_cpo_trust above."""
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)

    uniform_share = 1.0 / n_evs
    suspicious_share = uniform_share * suspicion_multiple

    for key, g in _iter_groups(df, group_cols):
        order = g.sort_values("connection_time").index
        ev = g.loc[order, "ev_id"].to_numpy()

        running_n: dict = {}
        total_seen = 0
        local = np.empty(len(g))
        for i, e in enumerate(ev):
            this_ev_n = running_n.get(e, 0)
            share_so_far = (this_ev_n / total_seen) if total_seen > 0 else 0.0  # cold start: trustworthy
            local[i] = float(np.clip(1.0 - share_so_far / suspicious_share, 0.0, 1.0))
            running_n[e] = this_ev_n + 1
            total_seen += 1
        trust.loc[order] = local

    df["naive_ev_trust"] = trust
    return df


def add_all_naive_trust(df: pd.DataFrame, cpo_trust_scale, group_cols=DEFAULT_GROUP_COLS) -> pd.DataFrame:
    df = add_naive_cpo_trust(df, cpo_trust_scale, group_cols=group_cols)
    df = add_naive_emsp_trust(df)
    df = add_naive_ev_trust(df, group_cols=group_cols)
    return df


In [ ]:
%%writefile delegation_baseline.py
'''
delegation_baseline.py
-------------------------
Demonstrates the delegation blind spot: a baseline that only tracks HOME
identity (cpo_id / emsp_id / ev_id -- what naive_trust.py, rbac_baseline.py
and abac_baseline.py already compute trust against) cannot see delegate-
side fraud, because the delegate's bad behavior never touches the home
identity's own trust history.

Two views, computed the SAME causal, (variant, world_id)-grouped way as
naive_trust.py (see that module's BUG LOG for why grouping matters):
  - naive_cpo_trust / naive_emsp_trust / naive_ev_trust (naive_trust.py):
    keyed on the HOME identity -- blind to delegation by construction.
  - delegate_cpo_trust / delegate_emsp_trust / delegate_breadth_score
    (NEW, this module): the SAME causal mechanism, keyed on the ACTING
    delegate identity instead.
'''

import numpy as np
import pandas as pd
from naive_trust import _iter_groups, _present_group_cols, DEFAULT_GROUP_COLS


def add_delegate_cpo_trust(df, scale, group_cols=DEFAULT_GROUP_COLS):
    '''Causal running-mean trust keyed on station_operator_id (not
    cpo_id). scale accepts the same float-or-{variant: float} shape as
    naive_trust.compute_cpo_trust_scale.'''
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)
    for key, g in _iter_groups(df, group_cols):
        if isinstance(scale, dict):
            variant_cols = _present_group_cols(df, ('variant',))
            this_scale = scale[g['variant'].iloc[0]] if variant_cols else next(iter(scale.values()))
        else:
            this_scale = scale

        order = g.sort_values('connection_time').index
        bd = g.loc[order, 'billing_deviation'].to_numpy()
        operator = g.loc[order, 'station_operator_id'].to_numpy()

        running_sum = {}
        running_n = {}
        local = np.empty(len(g))
        for i, c in enumerate(operator):
            n = running_n.get(c, 0)
            running_mean = (running_sum.get(c, 0.0) / n) if n > 0 else 0.0
            local[i] = float(np.clip(1.0 - running_mean / this_scale, 0.0, 1.0))
            running_sum[c] = running_sum.get(c, 0.0) + bd[i]
            running_n[c] = n + 1
        trust.loc[order] = local
    df['delegate_cpo_trust'] = trust
    return df


def add_delegate_emsp_trust(df, group_cols=DEFAULT_GROUP_COLS):
    '''Causal running dispute-rate trust keyed on issuing_emsp_id (not
    emsp_id), derived from delegate_dispute_flag rather than reusing the
    home-eMSP column -- same structure as naive_trust.add_naive_emsp_trust's
    1 - previous_flag_rate idea.'''
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)
    for key, g in _iter_groups(df, group_cols):
        order = g.sort_values('connection_time').index
        flag = g.loc[order, 'delegate_dispute_flag'].astype(float).to_numpy()
        issuer = g.loc[order, 'issuing_emsp_id'].to_numpy()

        running_sum = {}
        running_n = {}
        local = np.empty(len(g))
        for i, e in enumerate(issuer):
            n = running_n.get(e, 0)
            prior_rate = (running_sum.get(e, 0.0) / n) if n > 0 else 0.0
            local[i] = 1.0 - prior_rate
            running_sum[e] = running_sum.get(e, 0.0) + flag[i]
            running_n[e] = n + 1
        trust.loc[order] = local
    df['delegate_emsp_trust'] = trust
    return df


def add_delegate_breadth_score(df, group_cols=DEFAULT_GROUP_COLS, suspicious_breadth=3):
    '''EV delegation abuse is about BREADTH (how many DISTINCT owner
    ev_id accounts a given delegate_driver_id has touched so far), not
    raw session count -- a legitimate delegate normally serves ONE
    owner. Causal running count of distinct ev_id values seen under each
    delegate_driver_id so far.'''
    df = df.copy()
    trust = pd.Series(np.nan, index=df.index, dtype=float)
    for key, g in _iter_groups(df, group_cols):
        order = g.sort_values('connection_time').index
        delegate = g.loc[order, 'delegate_driver_id'].to_numpy()
        owner = g.loc[order, 'ev_id'].to_numpy()

        seen_owners = {}
        local = np.empty(len(g))
        for i, d in enumerate(delegate):
            owners_so_far = seen_owners.setdefault(d, set())
            breadth = len(owners_so_far)
            local[i] = float(np.clip(1.0 - breadth / suspicious_breadth, 0.0, 1.0))
            owners_so_far.add(owner[i])
        trust.loc[order] = local
    df['delegate_breadth_score'] = trust
    return df


def summarize_delegation_blind_spot(df):
    '''Of sessions genuinely run by a malicious delegate (each axis
    checked independently), how trustworthy does the HOME-identity view
    still think they are, versus the delegation-aware view?'''
    cpo_risky = df[df['gt_cpo_delegate_malicious'] == 1]
    print('CPO delegation -- sessions actually run by the malicious delegate operator:', len(cpo_risky), '/', len(df))
    if len(cpo_risky):
        home = cpo_risky['naive_cpo_trust'].mean()
        aware = cpo_risky['delegate_cpo_trust'].mean()
        print('  Home-identity view   (naive_cpo_trust,   blind to delegation): mean=%.3f' % home)
        print('  Delegation-aware view (delegate_cpo_trust):                   mean=%.3f' % aware)

    emsp_risky = df[df['gt_emsp_delegate_malicious'] == 1]
    print()
    print('eMSP delegation -- sessions actually issued by the malicious delegate:', len(emsp_risky), '/', len(df))
    if len(emsp_risky):
        home = emsp_risky['naive_emsp_trust'].mean()
        aware = emsp_risky['delegate_emsp_trust'].mean()
        print('  Home-identity view   (naive_emsp_trust,  blind to delegation): mean=%.3f' % home)
        print('  Delegation-aware view (delegate_emsp_trust):                  mean=%.3f' % aware)

    ev_risky = df[df['gt_ev_delegate_abuse'] == 1]
    print()
    print('EV delegation -- sessions driven by a malicious (multi-account) delegate:', len(ev_risky), '/', len(df))
    if len(ev_risky):
        home = ev_risky['naive_ev_trust'].mean()
        aware = ev_risky['delegate_breadth_score'].mean()
        print('  Home-identity view   (naive_ev_trust, keyed on ev_id):         mean=%.3f' % home)
        print('  Delegation-aware view (delegate_breadth_score):               mean=%.3f' % aware)

## Step 2: load the generated data

Reads the combined CSV notebook 02 already produced -- this notebook does not regenerate anything itself. If this cell errors with a missing-file or missing-column error, run (or re-run) notebook 02 first.

In [ ]:
import os
import pandas as pd

# Same path multi_world_pipeline.OUT_DIR defaults to in notebook 02 -- hardcoded
# here (not imported) so this notebook doesn't depend on multi_world_pipeline.py
# being written to disk in this runtime; only the CSV it already wrote to Drive.
OUT_DIR = os.environ.get(
    "V2G_WORLDS_OUT_DIR",
    "/content/drive/MyDrive/V2G experiments/trustmesh_data/full_pull/worlds",
)
combined_path = os.path.join(OUT_DIR, "v2g_benchmark_multi_world.csv")
df = pd.read_csv(combined_path)

required_cols = ["station_operator_id", "issuing_emsp_id", "delegate_driver_id",
                  "gt_cpo_delegate_malicious", "gt_emsp_delegate_malicious", "gt_ev_delegate_abuse"]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, (
    f"{combined_path} is missing delegation columns {missing} -- this CSV predates the "
    f"delegation axis. Run notebook 02 (which now writes these columns) before this one."
)
print(f"Loaded {len(df)} rows from {combined_path} -- delegation columns present.")


## Step 3: the actual check -- demonstrate the blind spot

Calibrates `naive_cpo_trust`/`delegate_cpo_trust` and runs the comparison on the FULL combined dataset (not a held-out test split), printed per variant.

In [ ]:
import naive_trust
import delegation_baseline

cpo_trust_scale = naive_trust.compute_cpo_trust_scale(df, 'CPO-B', per_variant=True)
df = naive_trust.add_all_naive_trust(df, cpo_trust_scale)

delegate_cpo_scale = naive_trust.compute_cpo_trust_scale(
    # df already has its own 'cpo_id' column -- drop it before renaming
    # station_operator_id to that same name, or pandas ends up with TWO
    # columns both called 'cpo_id' and compute_cpo_trust_scale's
    # g["cpo_id"] != malicious_cpo comparison returns a 2-column frame
    # instead of a Series, which .loc then rejects ("Cannot index with
    # multidimensional key"). Confirmed by running this cell for real.
    df.drop(columns=['cpo_id']).rename(columns={'station_operator_id': 'cpo_id'}),
    'CPO-D-sub', per_variant=True
)
df = delegation_baseline.add_delegate_cpo_trust(df, scale=delegate_cpo_scale)
df = delegation_baseline.add_delegate_emsp_trust(df)
df = delegation_baseline.add_delegate_breadth_score(df)

for variant_name in sorted(df['variant'].unique()):
    print('=' * 70)
    print('variant:', variant_name)
    print('=' * 70)
    delegation_baseline.summarize_delegation_blind_spot(df[df['variant'] == variant_name])
    print()

## Done

New columns on every row of `v2g_benchmark_multi_world.csv` (written by notebook 02): `station_operator_id`, `issuing_emsp_id`, `delegate_driver_id`, `is_delegated_session`, `delegate_dispute_flag`, `gt_cpo_delegate_malicious`, `gt_emsp_delegate_malicious`, `gt_ev_delegate_abuse`, `delegation_risk_factor_count`, `any_delegation_risk`.

Nothing in `benchmark_labels.py` needed to change -- these are generated directly during generation, the same way `authority_malicious`/`ev_sybil` already are, and are kept separate from `risk_factor_count`/`composite_risk_score`/`any_risk` on purpose.

Paste back: the Step 3 blind-spot comparison printed per variant.